In [0]:
from datetime import datetime
import os.path

today=datetime.now().strftime("%Y%m%d")

youtube_path = "/Volumes/workspace/default/youtube"
full_path=f"{youtube_path}/youtube_videos_{today}.parquet"

print(f"🔍 Checking for uploaded files for date: {today} ")
print("\n📁 YouTube files:")
try:
    exists = os.path.isfile(full_path)
    if exists:
        print("✅ Found YouTube data")
    else:
        print("❌ No YouTube data found")
except Exception as e:
    print(f"Error when trying to find youtube data: {e}")
    exit(-1)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_youtube_table = "workspace.default.bronze_youtube_videos"

def load_youtube_to_bronze():
    try:
        youtube_raw = spark.read.parquet(youtube_path)
        print(f"📊 Raw YouTube files loaded: {youtube_raw.count():,} records")

        youtube_bronze = youtube_raw \
            .withColumn("bronze_load_timestamp", current_timestamp()) 
            .withColumn("data_source", lit("youtube_api"))

        youtube_bronze.write.mode("overwrite").saveAsTable(bronze_youtube_table)
        
        final_count = spark.table(bronze_youtube_table).count()
        print(f"✅ YouTube Bronze table created: {final_count:,} records")
        
        # Show sample
        print("\n📺 Sample YouTube Bronze records:")
        spark.table(bronze_youtube_table).select(
            "title", "channel_title", "view_count", "like_count", "bronze_load_timestamp"
        ).show(3, truncate=False)
        
        return True
        
    except Exception as e:
        print(f"Error loading YouTube files: {e}")

In [0]:
youtube_data_loaded = load_youtube_to_bronze()

In [0]:
%run ../utils/bronze_ingestion_data_quality_report

In [0]:
if youtube_data_loaded:
    bronze_data_quality_report(bronze_youtube_table, "video_id",  ["video_id", "title", "channel_title"])